# Complete three-state figure at $K=40$ and $T=50$

This notebook generates the corrected six-panel figure at $K=40$ and $T=50$. Blue/red always denote negative/positive intrinsic bias. The lower row shows a stacked bias-coloured histogram, one KDE for the whole non-stubborn population ($h=0.075$), solid group MC means at $T=50$, and dashed bias-group MF predictions.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if not (repo_root / 'src').exists():
    raise RuntimeError('Open this notebook from the repository root or notebooks folder.')
sys.path.insert(0, str(repo_root))

from src.monte_carlo import MonteCarloParameters, simulate_trajectory
from src.initial_condition_states import simulate_asymmetric_bimodal

audit_script_dir = repo_root.parent
if not (audit_script_dir / 'audit_panel_alignment.py').exists():
    raise RuntimeError('The project panel-alignment audit helper is required.')
sys.path.insert(0, str(audit_script_dir))
from audit_panel_alignment import require_matplotlib_panel_alignment

figure_dir = repo_root / 'figures'
figure_dir.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8.0,
    'axes.labelsize': 12.0,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'legend.fontsize': 7.0,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.70,
    'ytick.major.width': 0.70,
    'xtick.major.size': 3.0,
    'ytick.major.size': 3.0,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

## 1. Parameters

In [ ]:
common = dict(
    n=800, k=40, t_max=50, f_s=0.50,
    sigma_s=0.80, sigma_ns=0.10, gamma=4.0, d_c=0.50,
    x0_stubborn_positive=1.0, x0_stubborn_negative=-1.0,
    trajectory_seed=123, trajectories_per_bias_group=200,
)

bp_parameters = MonteCarloParameters(
    **common, seed=80,
    x0_nonstubborn_negative=-0.5, x0_nonstubborn_positive=0.5,
)
bu_parameters = MonteCarloParameters(
    **common, seed=80,
    x0_nonstubborn_negative=0.5, x0_nonstubborn_positive=0.5,
)
ab_parameters = MonteCarloParameters(
    **common, seed=62,
    x0_nonstubborn_negative=-0.5, x0_nonstubborn_positive=0.5,
)

cases = [
    {'panel': '(a)', 'key': 'bp', 'name': 'Bimodal polarized',
     'negative_mf': -0.7518939448845263, 'positive_mf': 0.7518939448845263,
     'distribution_color': '#6F4E9C'},
    {'panel': '(b)', 'key': 'bu', 'name': 'Biased unimodal',
     'negative_mf': 0.5514161221836354, 'positive_mf': 0.6679211791156909,
     'distribution_color': '#D9772B'},
    {'panel': '(c)', 'key': 'ab', 'name': 'Asymmetric bimodal polarized',
     'negative_mf': -0.0714191175, 'positive_mf': 0.7233891075,
     'distribution_color': '#6F4E9C'},
]
bp_parameters, bu_parameters, ab_parameters

## 2. Run all three simulations

In [ ]:
results = {
    'bp': simulate_trajectory(alpha2=0.03, parameters=bp_parameters),
    'bu': simulate_trajectory(alpha2=0.03, parameters=bu_parameters),
    'ab': simulate_asymmetric_bimodal(alpha2=0.03, parameters=ab_parameters),
}

expected_initial_means = {
    'bp': (-0.5, 0.5),
    'bu': (0.5, 0.5),
    'ab': (0.0, 0.5),
}
for key, (expected_negative, expected_positive) in expected_initial_means.items():
    result = results[key]
    assert np.isclose(result['negative_mean'][0], expected_negative)
    assert np.isclose(result['positive_mean'][0], expected_positive)
cases[2]['negative_mf'] = results['ab']['negative_mf']
cases[2]['positive_mf'] = results['ab']['positive_mf']

for case in cases:
    result = results[case['key']]
    print(
        f"{case['name']}: N={result['parameters'].n}, "
        f"K={result['parameters'].k:g}, T={result['parameters'].t_max}, "
        f"seed={result['parameters'].seed}, B={result['B']:.3f}, S={result['S']:.3f}"
    )

## 3. Plotting functions

In [ ]:
negative_mc_color = '#2166AC'
positive_mc_color = '#B2182B'
negative_mf_color = '#00A6D6'
positive_mf_color = '#EF3B6B'
negative_trace_color = '#92C5DE'
positive_trace_color = '#F4A6B6'
kde_bandwidth = 0.075
show_histogram = True

def style_axis(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
        spine.set_color('#202020')
    ax.tick_params(direction='out', top=False, right=False)
    ax.grid(False)

def draw_trajectory(ax, result, case):
    is_negative = result['sampled_is_negative']
    ax.plot(
        result['times'], result['sampled_history'][:, is_negative],
        color=negative_trace_color, alpha=0.10, linewidth=0.35, rasterized=True,
    )
    ax.plot(
        result['times'], result['sampled_history'][:, ~is_negative],
        color=positive_trace_color, alpha=0.10, linewidth=0.35, rasterized=True,
    )
    ax.plot(result['times'], result['negative_mean'], color=negative_mc_color, linewidth=1.25)
    ax.plot(result['times'], result['positive_mean'], color=positive_mc_color, linewidth=1.25)
    ax.axhline(case['negative_mf'], color=negative_mf_color, linewidth=1.05, linestyle='--')
    ax.axhline(case['positive_mf'], color=positive_mf_color, linewidth=1.05, linestyle='--')
    ax.set(xlim=(0, 50), ylim=(-1.02, 1.02), xlabel=r'$t$', ylabel=r'$x_i(t)$')
    ax.set_xticks([0, 10, 20, 30, 40, 50])
    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    style_axis(ax)

def draw_distribution(ax, result, case):
    x_grid = np.linspace(-1.0, 1.0, 1000)
    bins = np.linspace(-1.0, 1.0, 31)
    bin_width = bins[1] - bins[0]
    if show_histogram:
        negative_values = result['final_negative_opinions']
        positive_values = result['final_positive_opinions']
        total_count = len(negative_values) + len(positive_values)
        negative_weights = np.full(len(negative_values), 1.0 / (total_count * bin_width))
        positive_weights = np.full(len(positive_values), 1.0 / (total_count * bin_width))
        ax.hist(
            [negative_values, positive_values], bins=bins,
            weights=[negative_weights, positive_weights], stacked=True,
            color=[negative_mc_color, positive_mc_color], alpha=0.32,
            edgecolor='none',
        )

    overall_values = result['final_opinions']
    overall_std = np.std(overall_values, ddof=1)
    if not np.isfinite(overall_std) or overall_std <= 0.0:
        raise ValueError('KDE requires a finite, non-zero terminal standard deviation.')
    overall_kde = gaussian_kde(
        overall_values, bw_method=kde_bandwidth / overall_std,
    )
    overall_density = overall_kde(x_grid)
    density_color = case['distribution_color']
    ax.fill_between(x_grid, 0.0, overall_density, color=density_color, alpha=0.10, linewidth=0)
    ax.plot(x_grid, overall_density, color=density_color, linewidth=1.35)
    ax.axvline(result['negative_mean'][-1], color=negative_mc_color, linewidth=1.25)
    ax.axvline(result['positive_mean'][-1], color=positive_mc_color, linewidth=1.25)
    ax.axvline(case['negative_mf'], color=negative_mf_color, linewidth=1.05, linestyle='--')
    ax.axvline(case['positive_mf'], color=positive_mf_color, linewidth=1.05, linestyle='--')
    ax.set(xlim=(-1.0, 1.0), ylim=(0, None), xlabel=r'$x_i$', ylabel=r'$p(x_i)$')
    ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    style_axis(ax)

legend_handles = [
    Line2D([0], [0], color=negative_mc_color, lw=1.25, label='Negative-bias MC mean'),
    Line2D([0], [0], color=positive_mc_color, lw=1.25, label='Positive-bias MC mean'),
    Line2D([0], [0], color=negative_mf_color, lw=1.05, ls='--', label='Negative-bias MF'),
    Line2D([0], [0], color=positive_mf_color, lw=1.05, ls='--', label='Positive-bias MF'),
]

## 4. Generate the three trajectory panels

In [ ]:
fig, axes = plt.subplots(
    1, 3, figsize=(12.8, 4.0), dpi=300,
    gridspec_kw={'left': 0.065, 'right': 0.985, 'bottom': 0.16, 'top': 0.74, 'wspace': 0.31},
)
for ax, case in zip(axes, cases):
    draw_trajectory(ax, results[case['key']], case)
    ax.set_title(case['name'], fontsize=15, fontweight='semibold', pad=9)
    ax.text(-0.15, 1.13, case['panel'], transform=ax.transAxes, fontsize=15, fontweight='bold')
fig.legend(
    handles=legend_handles, loc='upper center', ncol=4, frameon=True,
    bbox_to_anchor=(0.5, 0.94), fontsize=13.5, handlelength=3.0, columnspacing=1.35,
)
trajectory_base = figure_dir / 'three_states_K40_T50_seed62_trajectories'
require_matplotlib_panel_alignment(
    fig, json_out=str(trajectory_base) + '.alignment.json',
    overlay_svg=str(trajectory_base) + '.alignment.svg',
    tolerance_pt=1.5, gutter_tolerance_pt=1.5, strict=True,
)
fig.savefig(str(trajectory_base) + '.svg', bbox_inches='tight')
fig.savefig(str(trajectory_base) + '.pdf', bbox_inches='tight')
fig.savefig(str(trajectory_base) + '.png', dpi=600, bbox_inches='tight')
plt.show()

## 5. Generate the three terminal-distribution panels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.5), dpi=300)
for ax, case in zip(axes, cases):
    result = results[case['key']]
    draw_distribution(ax, result, case)
    ax.set_title(
        f"{case['name']}\n" + rf"$B={result['B']:.3f},\quad S={result['S']:.3f}$",
        fontsize=14, fontweight='semibold', pad=7,
    )
fig.tight_layout()
distribution_base = figure_dir / 'three_states_K40_T50_seed62_histogram_distribution_h075'
require_matplotlib_panel_alignment(
    fig, json_out=str(distribution_base) + '.alignment.json',
    overlay_svg=str(distribution_base) + '.alignment.svg',
    tolerance_pt=1.5, gutter_tolerance_pt=1.5, strict=True,
)
fig.savefig(str(distribution_base) + '.svg', bbox_inches='tight')
fig.savefig(str(distribution_base) + '.pdf', bbox_inches='tight')
fig.savefig(str(distribution_base) + '.png', dpi=600, bbox_inches='tight')
plt.show()

## 6. Generate the complete six-panel figure

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(7.20, 4.70), dpi=300,
    gridspec_kw={'height_ratios': [1.0, 0.78]},
)
fig.subplots_adjust(
    left=0.075, right=0.992, bottom=0.115, top=0.715,
    wspace=0.34, hspace=0.56,
)
for column, case in enumerate(cases):
    result = results[case['key']]
    draw_trajectory(axes[0, column], result, case)
    draw_distribution(axes[1, column], result, case)
    axes[0, column].text(
        0.5, 1.10, case['name'], transform=axes[0, column].transAxes,
        ha='center', va='bottom', fontsize=9.0, fontweight='bold',
    )
    axes[0, column].text(
        -0.14, 1.10, case['panel'], transform=axes[0, column].transAxes,
        ha='left', va='bottom', fontsize=10.5, fontweight='bold', clip_on=False,
    )
    axes[1, column].text(
        0.5, 1.015, rf"$B={result['B']:.3f},\quad S={result['S']:.3f}$",
        transform=axes[1, column].transAxes, ha='center', va='bottom',
        fontsize=8.0, clip_on=False,
    )
legend = fig.legend(
    handles=legend_handles, loc='upper center', ncol=4, frameon=True,
    bbox_to_anchor=(0.5, 0.82), fontsize=6.2, handlelength=2.2,
    handletextpad=0.45, columnspacing=0.9, borderpad=0.35,
    fancybox=False, framealpha=1.0,
)
legend.get_frame().set_edgecolor('#8A8A8A')
legend.get_frame().set_linewidth(0.55)
complete_base = figure_dir / 'three_states_K40_T50_seed62_complete_histogram_reference_type_h075'
require_matplotlib_panel_alignment(
    fig, json_out=str(complete_base) + '.alignment.json',
    overlay_svg=str(complete_base) + '.alignment.svg',
    tolerance_pt=1.5, gutter_tolerance_pt=1.5, strict=True,
)
fig.savefig(str(complete_base) + '.svg', bbox_inches='tight')
fig.savefig(str(complete_base) + '.pdf', bbox_inches='tight')
fig.savefig(str(complete_base) + '.png', dpi=600, bbox_inches='tight')
plt.show()